
## Fase 5 — Agregación por localidad y construcción de H1′

## Configuración

In [2]:
from pathlib import Path
import unicodedata
import re
import warnings

import numpy as np
import pandas as pd
from scipy.stats import spearmanr

np.random.seed(2026)
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

PROJECT_ROOT = Path("../..")

RUTAS = {
    "encuesta": PROJECT_ROOT / "outputs" / "encuesta_percepcion_legible.csv",
    "linea_purpura": PROJECT_ROOT / "outputs" / "lineapurpura.csv",
    "duplas": PROJECT_ROOT / "outputs" / "duplas.csv",
    "riesgo_feminicidio": PROJECT_ROOT / "outputs" / "riesgofeminicidio.csv",
}

RUTA_SUPUESTOS = Path("../../docs/supuestos.md")
RUTA_SUPUESTOS.parent.mkdir(parents=True, exist_ok=True)

for nombre, ruta in RUTAS.items():
    if not ruta.exists():
        raise FileNotFoundError(f"No se encontró '{nombre}': {ruta}")

print("Configuración cargada correctamente.")

Configuración cargada correctamente.


In [ ]:
def registrar_resultado(texto):
    with open(RUTA_SUPUESTOS, "a", encoding="utf-8") as f:
        f.write("\n" + texto.strip() + "\n")

def leer_csv_robusto(path):
    errores = []

    for encoding in ("utf-8-sig", "utf-8", "latin-1", "cp1252"):
        try:
            return pd.read_csv(
                path,
                sep=None,
                engine="python",
                encoding=encoding
            )
        except Exception as e:
            errores.append((encoding, str(e)))

    raise RuntimeError(
        f"No fue posible leer {path.name}. Intentos: {errores}"
    )

encuesta = leer_csv_robusto(RUTAS["encuesta"])
linea_purpura = leer_csv_robusto(RUTAS["linea_purpura"])
duplas = leer_csv_robusto(RUTAS["duplas"])
riesgo = leer_csv_robusto(RUTAS["riesgo_feminicidio"])

print("Fuentes cargadas correctamente.")

RuntimeError: No fue posible leer encuesta_percepcion_legible.csv. Intentos: [('utf-8-sig', "The 'low_memory' option is not supported with the 'python' engine"), ('utf-8', "The 'low_memory' option is not supported with the 'python' engine"), ('latin-1', "The 'low_memory' option is not supported with the 'python' engine"), ('cp1252', "The 'low_memory' option is not supported with the 'python' engine")]

## 5.1 — Indicadores poblacionales por localidad

In [ ]:
COLUMNAS_ENCUESTA = [
    "codigo_localidad",
    "codigo_UPL",
    "fexp_calp_anu",
    "fexp_calh_anu",
    "D1",
    "sexo_jefe",
    "C303",
    "ind_salud_102",
    "IPSJ_A",
    "IPSJ_C",
    "IPSJ_E",
    "ICG_B",
    "Mx404_1",
    "Mx404_2",
    "Mx404_3",
    "Mx404_4",
    "Mx404_5",
    "Mx404_6",
    "Ax201",
    "Bx201",
    "Cx201",
    "Dx201",
    "Ex201",
    "Fx201",
    "Gx201",
    "Hx201",
    "Ix201",
    "Jx201",
]

faltantes = [c for c in COLUMNAS_ENCUESTA if c not in encuesta.columns]
assert not faltantes, f"Faltan columnas en la encuesta: {faltantes}"

encuesta["codigo_localidad"] = pd.to_numeric(
    encuesta["codigo_localidad"], errors="raise"
).astype(int)

encuesta["codigo_UPL"] = pd.to_numeric(
    encuesta["codigo_UPL"], errors="raise"
).astype(int)

assert len(encuesta) == 13082
assert encuesta["codigo_localidad"].nunique() == 19
assert encuesta["codigo_UPL"].nunique() == 30
assert encuesta["codigo_localidad"].between(1, 19).all()

In [ ]:
cols_m_1a5 = [f"Mx404_{i}" for i in range(1, 6)]

encuesta["afronto_M"] = (
    encuesta[cols_m_1a5].eq("Si").any(axis=1)
).astype(int)

inconsistencia_404 = (
    encuesta["afronto_M"]
    != encuesta["Mx404_6"].eq("No").astype(int)
).sum()

componentes_iba = ["IPSJ_A", "IPSJ_C", "IPSJ_E"]
mask_iba = encuesta[componentes_iba].notna().all(axis=1)

encuesta["IBA"] = np.nan
z_componentes = []

for col in componentes_iba:
    serie = pd.to_numeric(encuesta.loc[mask_iba, col], errors="coerce")
    z = (serie - serie.mean()) / serie.std(ddof=0)
    z_componentes.append(z)

encuesta.loc[mask_iba, "IBA"] = -(
    z_componentes[0] + z_componentes[1] + z_componentes[2]
) / 3

categorias_gad_severas = {
    "Se aprecian síntomas de ansiedad moderados",
    "Se aprecian síntomas de ansiedad severos",
}

encuesta["gad7_mod_sev"] = (
    encuesta["ind_salud_102"].isin(categorias_gad_severas)
).astype(int)

encuesta["pobreza_subjetiva"] = (
    encuesta["C303"].eq("Si")
).astype(int)

In [ ]:
TAREAS = [
    "Ax201", "Bx201", "Cx201", "Dx201", "Ex201",
    "Fx201", "Gx201", "Hx201", "Ix201", "Jx201"
]

def calcular_icc_fila(row):
    respuestas = [
        row[c] for c in TAREAS
        if pd.notna(row[c]) and row[c] != "No se realiza"
    ]

    n_tareas = len(respuestas)

    if n_tareas < 5:
        return pd.Series(
            [np.nan, np.nan, np.nan, np.nan],
            index=[
                "HHI",
                "responsable_principal",
                "sexo_responsable_principal",
                "ICC_mujer",
            ],
        )

    conteos = pd.Series(respuestas).value_counts()
    proporciones = conteos / n_tareas
    hhi = float((proporciones ** 2).sum())
    responsable = conteos.idxmax()

    sexo = np.nan

    if responsable == "El /la jefe/a de hogar":
        if row["sexo_jefe"] in {"Mujer", "Hombre"}:
            sexo = row["sexo_jefe"]

    elif responsable == "El/la cónyuge o pareja del jefe de hogar":
        if row["sexo_jefe"] == "Hombre":
            sexo = "Mujer"
        elif row["sexo_jefe"] == "Mujer":
            sexo = "Hombre"

    icc_mujer = hhi if sexo == "Mujer" else 0.0

    return pd.Series(
        [hhi, responsable, sexo, icc_mujer],
        index=[
            "HHI",
            "responsable_principal",
            "sexo_responsable_principal",
            "ICC_mujer",
        ],
    )

icc = encuesta.apply(calcular_icc_fila, axis=1)
encuesta = pd.concat([encuesta, icc], axis=1)

encuesta["carga_concentrada_mujer"] = (
    encuesta["ICC_mujer"] >= 0.50
).astype("Int64")

In [ ]:
def estadistico_ponderado(data, variable, peso, grupo="codigo_localidad"):
    filas = []

    for codigo, g in data.groupby(grupo):
        sub = g[[variable, peso]].dropna().copy()

        x = pd.to_numeric(sub[variable], errors="coerce")
        w = pd.to_numeric(sub[peso], errors="coerce")

        valid = x.notna() & w.notna() & (w > 0)
        x = x[valid].to_numpy(dtype=float)
        w = w[valid].to_numpy(dtype=float)

        n = len(x)

        if n == 0:
            filas.append({
                grupo: codigo,
                "estimacion": np.nan,
                "se": np.nan,
                "ic_inf": np.nan,
                "ic_sup": np.nan,
                "n": 0,
                "n_eff": 0,
                "cv_pct": np.nan,
            })
            continue

        estimacion = np.average(x, weights=w)
        n_eff = (w.sum() ** 2) / np.square(w).sum()
        var_pond = np.average(np.square(x - estimacion), weights=w)
        se = np.sqrt(var_pond / n_eff) if n_eff > 0 else np.nan

        filas.append({
            grupo: codigo,
            "estimacion": estimacion,
            "se": se,
            "ic_inf": estimacion - 1.96 * se,
            "ic_sup": estimacion + 1.96 * se,
            "n": n,
            "n_eff": n_eff,
            "cv_pct": abs(se / estimacion) * 100
            if np.isfinite(estimacion) and estimacion != 0
            else np.nan,
        })

    return pd.DataFrame(filas)

def extraer_indicador(data, variable, peso, prefijo):
    out = estadistico_ponderado(data, variable, peso)

    return out.rename(columns={
        "estimacion": prefijo,
        "se": f"{prefijo}_se",
        "ic_inf": f"{prefijo}_ic_inf",
        "ic_sup": f"{prefijo}_ic_sup",
        "n": f"{prefijo}_n",
        "n_eff": f"{prefijo}_n_eff",
        "cv_pct": f"{prefijo}_cv_pct",
    })

indicadores = pd.DataFrame({
    "codigo_localidad": sorted(encuesta["codigo_localidad"].unique())
})

especificaciones = [
    ("afronto_M", "fexp_calp_anu", "TAC_M"),
    ("carga_concentrada_mujer", "fexp_calh_anu", "pct_carga_mujer"),
    ("gad7_mod_sev", "fexp_calp_anu", "gad7_mod_sev"),
    ("pobreza_subjetiva", "fexp_calp_anu", "pobreza_subjetiva"),
    ("IPSJ_C", "fexp_calp_anu", "IPSJ_C_promedio"),
    ("ICG_B", "fexp_calp_anu", "ICG_B_promedio"),
    ("IBA", "fexp_calp_anu", "IBA_promedio"),
]

for variable, peso, prefijo in especificaciones:
    tabla = extraer_indicador(encuesta, variable, peso, prefijo)
    indicadores = indicadores.merge(
        tabla,
        on="codigo_localidad",
        how="left"
    )

encuesta["es_mujer"] = encuesta["D1"].eq("Mujer")
mujeres = encuesta[encuesta["es_mujer"]].copy()

pob_adulta_mujeres = (
    mujeres.groupby("codigo_localidad")["fexp_calp_anu"]
    .sum()
    .rename("pob_mujeres_adultas_expandida")
    .reset_index()
)

vol_min = (
    mujeres.loc[mujeres["afronto_M"] == 1]
    .groupby("codigo_localidad")["fexp_calp_anu"]
    .sum()
    .rename("vol_mujeres_visible_afrontada_min")
    .reset_index()
)

indicadores = (
    indicadores
    .merge(pob_adulta_mujeres, on="codigo_localidad", how="left")
    .merge(vol_min, on="codigo_localidad", how="left")
)

indicadores["vol_mujeres_visible_afrontada_min"] = (
    indicadores["vol_mujeres_visible_afrontada_min"].fillna(0)
)

upl_localidad = (
    encuesta.groupby("codigo_localidad")["codigo_UPL"]
    .nunique()
    .rename("n_UPL")
    .reset_index()
)

indicadores = indicadores.merge(
    upl_localidad,
    on="codigo_localidad",
    how="left"
)

tac_ciudad = np.average(
    encuesta["afronto_M"],
    weights=encuesta["fexp_calp_anu"]
)

texto_resultado_5_1 = f"""
## Paso 5.1 — Indicadores poblacionales por localidad ({pd.Timestamp.now().strftime('%Y-%m-%d')})

**Resultados**

- Registros analizados: **{len(encuesta):,}**.
- Localidades con información de encuesta: **{encuesta['codigo_localidad'].nunique()}**.
- UPL presentes: **{encuesta['codigo_UPL'].nunique()}**.
- TAC_M ponderada para Bogotá: **{tac_ciudad*100:.1f}%**.
- Mujeres adultas expandidas en las 19 localidades: **{indicadores['pob_mujeres_adultas_expandida'].sum():,.0f}**.
- Volumen expandido mínimo de mujeres con violencia visible y afrontada: **{indicadores['vol_mujeres_visible_afrontada_min'].sum():,.0f}**.
- Registros válidos para IBA: **{encuesta['IBA'].notna().sum():,}**.
- Registros válidos para ICC: **{encuesta['HHI'].notna().sum():,}**.
- Inconsistencias observadas entre Mx404_1..5 y Mx404_6: **{inconsistencia_404}**.

**Conclusión**

Al menos **{tac_ciudad*100:.1f}%** de la población representada por la encuesta presenció y afrontó una situación de violencia contra una mujer. Este valor constituye una cota inferior de la violencia socialmente visible y afrontada y no una estimación de prevalencia total.
"""

registrar_resultado(texto_resultado_5_1)
print("Resultado del Paso 5.1 guardado en ../../docs/supuestos.md")

## 5.2 — Criterio de publicación

In [ ]:
INDICADORES_CV = [
    "TAC_M",
    "pct_carga_mujer",
    "gad7_mod_sev",
    "pobreza_subjetiva",
]

for prefijo in INDICADORES_CV:
    indicadores[f"{prefijo}_publicable"] = (
        (indicadores[f"{prefijo}_n_eff"] >= 30)
        & (indicadores[f"{prefijo}_cv_pct"] <= 30)
    )

for prefijo in [
    "IPSJ_C_promedio",
    "ICG_B_promedio",
    "IBA_promedio",
]:
    indicadores[f"{prefijo}_publicable"] = (
        indicadores[f"{prefijo}_n_eff"] >= 30
    )

indicadores["publicable_demanda_min"] = (
    indicadores["TAC_M_publicable"]
    & indicadores["vol_mujeres_visible_afrontada_min"].gt(0)
)

resumen_publicacion = pd.DataFrame({
    "indicador": [
        "TAC_M",
        "pct_carga_mujer",
        "gad7_mod_sev",
        "pobreza_subjetiva",
        "IPSJ_C_promedio",
        "ICG_B_promedio",
        "IBA_promedio",
    ],
    "localidades_publicables": [
        int(indicadores["TAC_M_publicable"].sum()),
        int(indicadores["pct_carga_mujer_publicable"].sum()),
        int(indicadores["gad7_mod_sev_publicable"].sum()),
        int(indicadores["pobreza_subjetiva_publicable"].sum()),
        int(indicadores["IPSJ_C_promedio_publicable"].sum()),
        int(indicadores["ICG_B_promedio_publicable"].sum()),
        int(indicadores["IBA_promedio_publicable"].sum()),
    ]
})

resumen_publicacion["localidades_no_publicables"] = (
    19 - resumen_publicacion["localidades_publicables"]
)

n_no_pub_demanda = int(
    (~indicadores["publicable_demanda_min"]).sum()
)

tabla_pub_md = "\n".join(
    f"- {row.indicador}: **{row.localidades_publicables}/19 localidades publicables**."
    for row in resumen_publicacion.itertuples()
)

texto_resultado_5_2 = f"""
## Paso 5.2 — Criterio de publicación ({pd.Timestamp.now().strftime('%Y-%m-%d')})

**Resultados**

{tabla_pub_md}

- Localidades no publicables para el denominador mínimo de demanda: **{n_no_pub_demanda}**.

**Conclusión**

Las estimaciones territoriales se conservan únicamente cuando cumplen los umbrales de precisión definidos: tamaño efectivo mínimo de 30 observaciones y coeficiente de variación máximo de 30% para proporciones. Las localidades que no cumplen estos criterios no deben presentarse como estimaciones territoriales válidas.
"""

registrar_resultado(texto_resultado_5_2)

if n_no_pub_demanda > 5:
    raise RuntimeError(
        "Más de cinco localidades no cumplen el criterio de publicación "
        "para el denominador de demanda mínima."
    )

print("Resultado del Paso 5.2 guardado en ../../docs/supuestos.md")

## 5.3 — Oferta institucional

In [ ]:
LOCALIDADES = {
    1: "Usaquén", 2: "Chapinero", 3: "Santa Fe", 4: "San Cristóbal",
    5: "Usme", 6: "Tunjuelito", 7: "Bosa", 8: "Kennedy",
    9: "Fontibón", 10: "Engativá", 11: "Suba", 12: "Barrios Unidos",
    13: "Teusaquillo", 14: "Los Mártires", 15: "Antonio Nariño",
    16: "Puente Aranda", 17: "La Candelaria",
    18: "Rafael Uribe Uribe", 19: "Ciudad Bolívar", 20: "Sumapaz",
}

def normalizar_texto(texto):
    if pd.isna(texto):
        return np.nan

    texto = str(texto).strip().upper()
    texto = "".join(
        c for c in unicodedata.normalize("NFD", texto)
        if unicodedata.category(c) != "Mn"
    )
    return re.sub(r"\s+", " ", texto)

MAPA_NOMBRE_CODIGO = {
    normalizar_texto(nombre): codigo
    for codigo, nombre in LOCALIDADES.items()
}

def buscar_columna(df, candidatos):
    normalizadas = {
        normalizar_texto(c): c
        for c in df.columns
    }

    for candidato in candidatos:
        clave = normalizar_texto(candidato)
        if clave in normalizadas:
            return normalizadas[clave]

    return None

def asegurar_codigo_localidad(df):
    out = df.copy()

    col_codigo = buscar_columna(
        out,
        ["codigo_localidad", "CODIGO_LOCALIDAD", "Cod_Locali"]
    )

    if col_codigo:
        out["codigo_localidad"] = pd.to_numeric(
            out[col_codigo], errors="coerce"
        )
    else:
        col_localidad = buscar_columna(
            out,
            ["Localidad", "LOCALIDAD", "nombre_localidad"]
        )

        if col_localidad is None:
            raise KeyError(
                "La fuente no contiene código ni nombre de localidad."
            )

        out["codigo_localidad"] = (
            out[col_localidad]
            .map(normalizar_texto)
            .map(MAPA_NOMBRE_CODIGO)
        )

    if out["codigo_localidad"].isna().any():
        raise ValueError(
            "Hay filas que no pudieron asociarse a una localidad."
        )

    out["codigo_localidad"] = out["codigo_localidad"].astype(int)
    assert out["codigo_localidad"].between(1, 20).all()

    return out

def preparar_fecha(df):
    out = df.copy()

    col_fecha = buscar_columna(
        out,
        ["Fecha", "FECHA", "fecha"]
    )

    if col_fecha is None:
        raise KeyError("No se encontró columna de fecha.")

    out["fecha_dt"] = pd.to_datetime(
        out[col_fecha],
        errors="coerce",
        dayfirst=True,
    )

    if out["fecha_dt"].isna().any():
        raise ValueError("Se encontraron fechas no interpretables.")

    out["periodo_mes"] = out["fecha_dt"].dt.to_period("M")

    return out

linea_purpura = preparar_fecha(
    asegurar_codigo_localidad(linea_purpura)
)

duplas = preparar_fecha(
    asegurar_codigo_localidad(duplas)
)

riesgo = preparar_fecha(
    asegurar_codigo_localidad(riesgo)
)

In [ ]:
col_lp_total = buscar_columna(
    linea_purpura,
    ["TotalAtenciones", "Total Atenciones"]
)

col_duplas_total = buscar_columna(
    duplas,
    ["TotalAtenciones", "Total Atenciones"]
)

assert col_lp_total is not None
assert col_duplas_total is not None

periodos_lp = set(linea_purpura["periodo_mes"].unique())
periodos_duplas = set(duplas["periodo_mes"].unique())
periodos_riesgo = set(riesgo["periodo_mes"].unique())

periodos_comunes = sorted(
    periodos_lp & periodos_duplas & periodos_riesgo
)

if len(periodos_comunes) != 4:
    raise ValueError(
        f"Se esperaban cuatro cortes comunes. Encontrados: {periodos_comunes}"
    )

lp_comun = linea_purpura[
    linea_purpura["periodo_mes"].isin(periodos_comunes)
].copy()

duplas_comun = duplas[
    duplas["periodo_mes"].isin(periodos_comunes)
].copy()

oferta_lp = (
    lp_comun.groupby("codigo_localidad")[col_lp_total]
    .sum()
    .rename("atenciones_linea_purpura")
    .reset_index()
)

oferta_duplas = (
    duplas_comun.groupby("codigo_localidad")[col_duplas_total]
    .sum()
    .rename("atenciones_duplas")
    .reset_index()
)

oferta = (
    pd.DataFrame({"codigo_localidad": range(1, 21)})
    .merge(oferta_lp, on="codigo_localidad", how="left")
    .merge(oferta_duplas, on="codigo_localidad", how="left")
    .fillna({
        "atenciones_linea_purpura": 0,
        "atenciones_duplas": 0,
    })
)

oferta["oferta_total"] = (
    oferta["atenciones_linea_purpura"]
    + oferta["atenciones_duplas"]
)

oferta_resultados = oferta.copy()
oferta_resultados["localidad"] = (
    oferta_resultados["codigo_localidad"].map(LOCALIDADES)
)

top_oferta = (
    oferta_resultados
    .nlargest(3, "oferta_total")
    [["localidad", "oferta_total"]]
)

top_oferta_md = "\n".join(
    f"- {r.localidad}: **{r.oferta_total:,.0f} atenciones**."
    for r in top_oferta.itertuples()
)

texto_resultado_5_3 = f"""
## Paso 5.3 — Oferta institucional ({pd.Timestamp.now().strftime('%Y-%m-%d')})

**Resultados**

- Cortes comunes analizados: **{", ".join(str(p) for p in periodos_comunes)}**.
- Atenciones acumuladas de Línea Púrpura: **{oferta['atenciones_linea_purpura'].sum():,.0f}**.
- Atenciones acumuladas de Duplas: **{oferta['atenciones_duplas'].sum():,.0f}**.
- Oferta institucional total: **{oferta['oferta_total'].sum():,.0f} atenciones**.
- Localidades con mayor oferta acumulada:
{top_oferta_md}

**Conclusión**

La oferta institucional se concentra de forma desigual entre localidades. La comparación territorial se realiza únicamente sobre los cuatro cortes comunes entre las fuentes, evitando inflar la cobertura con periodos sin comparador administrativo equivalente.
"""

registrar_resultado(texto_resultado_5_3)
print("Resultado del Paso 5.3 guardado en ../../docs/supuestos.md")

## 5.4 — Registro administrativo de riesgo

In [ ]:
col_riesgo_total = buscar_columna(riesgo, ["Total", "TOTAL"])
col_pob_mujeres = buscar_columna(
    riesgo,
    ["PobMujeres", "Pob Mujeres", "pob_mujeres"]
)

assert col_riesgo_total is not None
assert col_pob_mujeres is not None

riesgo_comun = riesgo[
    riesgo["periodo_mes"].isin(periodos_comunes)
].copy()

casos_admin = (
    riesgo_comun
    .groupby("codigo_localidad")[col_riesgo_total]
    .sum()
    .rename("casos_admin")
    .reset_index()
)

idx_ultimo = (
    riesgo.sort_values("fecha_dt")
    .groupby("codigo_localidad")["fecha_dt"]
    .idxmax()
)

poblacion_mujeres = (
    riesgo.loc[
        idx_ultimo,
        ["codigo_localidad", col_pob_mujeres]
    ]
    .rename(columns={
        col_pob_mujeres: "pob_mujeres_oficial"
    })
)

admin = (
    pd.DataFrame({"codigo_localidad": range(1, 21)})
    .merge(casos_admin, on="codigo_localidad", how="left")
    .merge(poblacion_mujeres, on="codigo_localidad", how="left")
)

assert admin["casos_admin"].notna().all()
assert admin["pob_mujeres_oficial"].notna().all()

admin["tasa_admin_100k"] = (
    admin["casos_admin"]
    / admin["pob_mujeres_oficial"]
    * 100000
)

oferta = oferta.merge(
    admin[["codigo_localidad", "pob_mujeres_oficial"]],
    on="codigo_localidad",
    how="left"
)

oferta["oferta_100k"] = (
    oferta["oferta_total"]
    / oferta["pob_mujeres_oficial"]
    * 100000
)

tabla_admin = admin.copy()
tabla_admin["localidad"] = (
    tabla_admin["codigo_localidad"].map(LOCALIDADES)
)

top_riesgo = (
    tabla_admin
    .nlargest(3, "tasa_admin_100k")
    [["localidad", "tasa_admin_100k", "casos_admin"]]
)

top_riesgo_md = "\n".join(
    f"- {r.localidad}: **{r.tasa_admin_100k:.1f} casos por 100.000 mujeres** "
    f"({r.casos_admin:,.0f} casos acumulados)."
    for r in top_riesgo.itertuples()
)

texto_resultado_5_4 = f"""
## Paso 5.4 — Registro administrativo de riesgo ({pd.Timestamp.now().strftime('%Y-%m-%d')})

**Resultados**

- Casos administrativos acumulados en los cuatro cortes comunes: **{admin['casos_admin'].sum():,.0f}**.
- Población femenina oficial utilizada como denominador: **{admin['pob_mujeres_oficial'].sum():,.0f}**.
- Localidades con mayor tasa administrativa:
{top_riesgo_md}

**Conclusión**

La tasa administrativa permite normalizar territorialmente los casos registrados por población femenina. Este indicador representa demanda capturada por el sistema y no debe interpretarse como prevalencia total de violencia.
"""

registrar_resultado(texto_resultado_5_4)
print("Resultado del Paso 5.4 guardado en ../../docs/supuestos.md")

In [ ]:
comparabilidad = indicadores[
    [
        "codigo_localidad",
        "pob_mujeres_adultas_expandida",
    ]
].merge(
    admin[
        [
            "codigo_localidad",
            "pob_mujeres_oficial",
        ]
    ],
    on="codigo_localidad",
    how="left"
)

comparabilidad = comparabilidad[
    comparabilidad["codigo_localidad"].between(1, 19)
].copy()

comparabilidad["dif_pct"] = (
    (
        comparabilidad["pob_mujeres_adultas_expandida"]
        - comparabilidad["pob_mujeres_oficial"]
    )
    / comparabilidad["pob_mujeres_oficial"]
    * 100
)

pob_adulta_ciudad = (
    comparabilidad["pob_mujeres_adultas_expandida"].sum()
)

pob_oficial_ciudad = (
    comparabilidad["pob_mujeres_oficial"].sum()
)

dif_ciudad = (
    (pob_adulta_ciudad - pob_oficial_ciudad)
    / pob_oficial_ciudad
    * 100
)

## 5.5 — Razones de cobertura

In [ ]:
fase5 = (
    indicadores
    .merge(
        oferta[
            [
                "codigo_localidad",
                "atenciones_linea_purpura",
                "atenciones_duplas",
                "oferta_total",
                "oferta_100k",
            ]
        ],
        on="codigo_localidad",
        how="left"
    )
    .merge(
        admin[
            [
                "codigo_localidad",
                "casos_admin",
                "pob_mujeres_oficial",
                "tasa_admin_100k",
            ]
        ],
        on="codigo_localidad",
        how="left"
    )
)

fase5 = fase5[
    fase5["codigo_localidad"].between(1, 19)
].copy()

fase5["localidad"] = (
    fase5["codigo_localidad"].map(LOCALIDADES)
)

fase5["RC_admin"] = (
    fase5["oferta_total"]
    / fase5["casos_admin"]
)

fase5["RC_real_cota_superior"] = (
    fase5["oferta_total"]
    / fase5["vol_mujeres_visible_afrontada_min"]
    * 1000
)

assert np.isfinite(fase5["RC_admin"]).all()
assert np.isfinite(fase5["RC_real_cota_superior"]).all()

mejor_admin = fase5.loc[fase5["RC_admin"].idxmax()]
peor_admin = fase5.loc[fase5["RC_admin"].idxmin()]
mejor_real = fase5.loc[fase5["RC_real_cota_superior"].idxmax()]
peor_real = fase5.loc[fase5["RC_real_cota_superior"].idxmin()]

texto_resultado_5_5 = f"""
## Paso 5.5 — Razones de cobertura ({pd.Timestamp.now().strftime('%Y-%m-%d')})

**Resultados**

- Mayor RC_admin: **{mejor_admin['localidad']}**, con **{mejor_admin['RC_admin']:.2f} atenciones por caso administrativo**.
- Menor RC_admin: **{peor_admin['localidad']}**, con **{peor_admin['RC_admin']:.2f} atenciones por caso administrativo**.
- Mayor cobertura frente a necesidad mínima observable: **{mejor_real['localidad']}**, con **{mejor_real['RC_real_cota_superior']:.2f} atenciones por cada 1.000 mujeres del volumen mínimo estimado**.
- Menor cobertura frente a necesidad mínima observable: **{peor_real['localidad']}**, con **{peor_real['RC_real_cota_superior']:.2f} atenciones por cada 1.000 mujeres del volumen mínimo estimado**.
- Mujeres adultas expandidas en las 19 localidades: **{pob_adulta_ciudad:,.0f}**.
- Población femenina oficial en las mismas localidades: **{pob_oficial_ciudad:,.0f}**.
- Diferencia relativa entre ambos universos poblacionales: **{dif_ciudad:.1f}%**.

**Conclusión**

RC_admin expresa cobertura frente a los casos capturados por el sistema. La segunda razón usa un denominador poblacional independiente, pero debe interpretarse como una **cota superior de cobertura**, porque la TAC solo identifica violencia visible y afrontada y no toda la necesidad real existente.
"""

registrar_resultado(texto_resultado_5_5)
print("Resultado del Paso 5.5 guardado en ../../docs/supuestos.md")

## 5.6 — Divergencia de rankings

In [ ]:
fase5["rank_admin"] = (
    fase5["RC_admin"]
    .rank(ascending=False, method="min")
    .astype(int)
)

fase5["rank_real"] = (
    fase5["RC_real_cota_superior"]
    .rank(ascending=False, method="min")
    .astype(int)
)

fase5["delta_rank"] = (
    fase5["rank_real"]
    - fase5["rank_admin"]
)

fase5["cambio_relevante"] = (
    fase5["delta_rank"].abs() >= 4
)

ranking = fase5[
    [
        "codigo_localidad",
        "localidad",
        "RC_admin",
        "RC_real_cota_superior",
        "rank_admin",
        "rank_real",
        "delta_rank",
        "cambio_relevante",
    ]
].sort_values(
    "delta_rank",
    ascending=False
)

n_cambio = int(fase5["cambio_relevante"].sum())

sobreestimadas = ranking[
    ranking["delta_rank"] >= 4
]

subestimadas = ranking[
    ranking["delta_rank"] <= -4
].sort_values("delta_rank")

sobre_md = (
    "\n".join(
        f"- {r.localidad}: pasa de posición **{r.rank_admin}** a **{r.rank_real}** "
        f"(Δ = **{r.delta_rank:+d}**)."
        for r in sobreestimadas.itertuples()
    )
    if len(sobreestimadas)
    else "- Ninguna localidad."
)

sub_md = (
    "\n".join(
        f"- {r.localidad}: pasa de posición **{r.rank_admin}** a **{r.rank_real}** "
        f"(Δ = **{r.delta_rank:+d}**)."
        for r in subestimadas.itertuples()
    )
    if len(subestimadas)
    else "- Ninguna localidad."
)

texto_resultado_5_6 = f"""
## Paso 5.6 — Divergencia de rankings ({pd.Timestamp.now().strftime('%Y-%m-%d')})

**Resultados**

- Localidades con cambio de al menos cuatro posiciones: **{n_cambio} de 19**.

Localidades que aparecen mejor cubiertas bajo el denominador administrativo que bajo el denominador poblacional mínimo:

{sobre_md}

Localidades que mejoran su posición al utilizar el denominador poblacional mínimo:

{sub_md}

**Conclusión**

La magnitud de `delta_rank` permite identificar qué localidades cambian sustancialmente de posición cuando la cobertura deja de evaluarse exclusivamente contra los casos que el propio sistema registró.
"""

registrar_resultado(texto_resultado_5_6)
print("Resultado del Paso 5.6 guardado en ../../docs/supuestos.md")

## 5.7 — Prueba estadística de H1′

In [ ]:
def spearman_bootstrap(x, y, n_boot=2000, semilla=2026):
    datos = pd.DataFrame({
        "x": x,
        "y": y
    }).dropna()

    x0 = datos["x"].to_numpy(dtype=float)
    y0 = datos["y"].to_numpy(dtype=float)

    rho, p = spearmanr(x0, y0)

    rng = np.random.default_rng(semilla)
    boot = []

    n = len(datos)

    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        r, _ = spearmanr(x0[idx], y0[idx])

        if np.isfinite(r):
            boot.append(r)

    ic_inf, ic_sup = np.percentile(boot, [2.5, 97.5])

    return {
        "rho": float(rho),
        "p": float(p),
        "ic_inf": float(ic_inf),
        "ic_sup": float(ic_sup),
        "n": n,
        "n_boot_validas": len(boot),
    }

pruebas = {
    "rank_admin_vs_rank_real": spearman_bootstrap(
        fase5["rank_admin"],
        fase5["rank_real"],
    ),
    "IPSJ_C_vs_delta_rank": spearman_bootstrap(
        fase5["IPSJ_C_promedio"],
        fase5["delta_rank"],
    ),
    "ICG_B_vs_delta_rank": spearman_bootstrap(
        fase5["ICG_B_promedio"],
        fase5["delta_rank"],
    ),
    "tasa_admin_vs_RC_real": spearman_bootstrap(
        fase5["tasa_admin_100k"],
        fase5["RC_real_cota_superior"],
    ),
}

r_rank = pruebas["rank_admin_vs_rank_real"]
r_ipsj = pruebas["IPSJ_C_vs_delta_rank"]
r_icg = pruebas["ICG_B_vs_delta_rank"]
r_admin = pruebas["tasa_admin_vs_RC_real"]

criterio_refutacion = (
    r_rank["rho"] > 0.85
    and r_rank["ic_inf"] > 0.60
)

estado_h1 = (
    "refutada por alta concordancia entre rankings"
    if criterio_refutacion
    else "no refutada por el criterio de alta concordancia"
)

signo_ipsj = (
    "coherente con la expectativa"
    if r_ipsj["rho"] < 0
    else "contrario a la expectativa"
)

signo_icg = (
    "coherente con la expectativa"
    if r_icg["rho"] < 0
    else "contrario a la expectativa"
)

signo_admin = (
    "negativo"
    if r_admin["rho"] < 0
    else "positivo"
)

texto_resultado_5_7 = f"""
## Paso 5.7 — Prueba estadística de H1′ ({pd.Timestamp.now().strftime('%Y-%m-%d')})

**Resultados**

- Concordancia entre `rank_admin` y `rank_real`: **ρ = {r_rank['rho']:.3f}**, IC95% **[{r_rank['ic_inf']:.3f}, {r_rank['ic_sup']:.3f}]**, p = **{r_rank['p']:.4f}**.
- Asociación entre `IPSJ_C` promedio y `delta_rank`: **ρ = {r_ipsj['rho']:.3f}**, IC95% **[{r_ipsj['ic_inf']:.3f}, {r_ipsj['ic_sup']:.3f}]**, p = **{r_ipsj['p']:.4f}**; signo **{signo_ipsj}**.
- Asociación entre `ICG_B` promedio y `delta_rank`: **ρ = {r_icg['rho']:.3f}**, IC95% **[{r_icg['ic_inf']:.3f}, {r_icg['ic_sup']:.3f}]**, p = **{r_icg['p']:.4f}**; signo **{signo_icg}**.
- Asociación entre tasa administrativa y cobertura frente a necesidad mínima: **ρ = {r_admin['rho']:.3f}**, IC95% **[{r_admin['ic_inf']:.3f}, {r_admin['ic_sup']:.3f}]**, p = **{r_admin['p']:.4f}**; signo **{signo_admin}**.
- Réplicas bootstrap utilizadas por prueba: hasta **2.000**.
- Resultado frente al criterio predefinido de refutación: H1′ queda **{estado_h1}**.

**Conclusión**

La evidencia territorial permite evaluar si el orden de cobertura derivado del registro administrativo coincide con el orden obtenido al utilizar un denominador poblacional independiente. La conclusión sobre H1′ se determina a partir de la concordancia de rankings y de la dirección de las asociaciones con acceso percibido a medios de denuncia, confianza vecinal y riesgo administrativo.
"""

registrar_resultado(texto_resultado_5_7)
print("Resultado del Paso 5.7 guardado en ../../docs/supuestos.md")

## Tabla consolidada

In [ ]:
columnas_finales = [
    "codigo_localidad",
    "localidad",
    "TAC_M",
    "TAC_M_ic_inf",
    "TAC_M_ic_sup",
    "pct_carga_mujer",
    "gad7_mod_sev",
    "pobreza_subjetiva",
    "IPSJ_C_promedio",
    "ICG_B_promedio",
    "IBA_promedio",
    "pob_mujeres_adultas_expandida",
    "vol_mujeres_visible_afrontada_min",
    "oferta_total",
    "oferta_100k",
    "casos_admin",
    "tasa_admin_100k",
    "RC_admin",
    "RC_real_cota_superior",
    "rank_admin",
    "rank_real",
    "delta_rank",
    "cambio_relevante",
    "publicable_demanda_min",
]

resultado_fase5_57 = (
    fase5[columnas_finales]
    .sort_values("rank_real")
    .reset_index(drop=True)
)

display(resultado_fase5_57)